In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

sns.set(style="whitegrid", context="notebook")

In [ ]:
train = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
X = train.drop("label", axis=1).values / 255.0
y = train["label"].values
test_data = test.values / 255.0

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(12, 4))
for i, ax in enumerate(axes):
    ax.imshow(X[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"Label: {y[i]}")
    ax.axis("off")
plt.show()

# Baseline logistic regression

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

lr = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    multi_class="multinomial",
    n_jobs=-1
)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Validation Set")
plt.show()

print(classification_report(y_val, y_pred))

precision, recall, f1, support = precision_recall_fscore_support(y_val, y_pred)
metrics_df = pd.DataFrame({
    "Digit": np.arange(10),
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "Support": support
})

metrics_melted = metrics_df.melt(
    id_vars="Digit",
    value_vars=["Precision", "Recall", "F1-score"],
    var_name="Metric", value_name="Score"
)

plt.figure(figsize=(12,6))
sns.barplot(data=metrics_melted, x="Digit", y="Score", hue="Metric")
plt.title("Precision, Recall and F1-score by Digit (Validation Set)")
plt.ylim(0,1.05)
plt.show()

# CNN

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

X = train.drop("label", axis=1).values.reshape(-1,28,28,1) / 255.0
y = keras.utils.to_categorical(train["label"].values, 10)
test_data = test.values.reshape(-1,28,28,1) / 255.0

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=train["label"].values
)

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)
datagen.fit(X_train)

model = keras.Sequential([
    layers.Conv2D(64, (3,3), activation="relu", input_shape=(28,28,1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True
    )
]

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=128),
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=2
)

In [ ]:
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation accuracy: {val_acc:.4f}")

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
submission = pd.DataFrame({
    "ImageId": list(range(1, len(test_data)+1)),
    "Label": np.argmax(model.predict(test_data), axis=1)
})
submission.to_csv("submission.csv", index=False)
submission.head()